<a href="https://colab.research.google.com/github/raviradadiya1414/pdf_chatbort/blob/main/pdf_chatbort.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain_community

In [ ]:
!pip install -U langchain-community pyp!df

ERROR: Invalid requirement: 'pyp!df': Expected end or semicolon (after name and no valid version specifier)
    pyp!df
       ^


In [ ]:
!pip install -qU langchain-groq

In [ ]:
!pip install langchain-chroma

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.vectorstores import FAISS
import os

In [ ]:
os.environ['GROQ_API_KEY'] = "api_key"

In [ ]:
loader = PyPDFLoader(
    file_path='/content/LLM.pdf'
)

In [ ]:
doc = loader.load()

In [ ]:
text_spliter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
data  = text_spliter.split_documents(doc)

In [ ]:
embadings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
!pip install faiss-gpu

In [ ]:
vector_store = FAISS.from_documents(data,embadings)

In [ ]:
llm = ChatGroq(
    model='llama-3.3-70b-versatile'
)

In [ ]:
prompt = ChatPromptTemplate.from_template(
    '''
    you are AI assitend.you only give question answer from context.you give brief answer.
    context:
    {context}
    question: {question}
    answer:
    '''
)


In [ ]:
store = {}
def get_message_history(session_id: str)->BaseChatMessageHistory:
  if session_id not in store:
    store[session_id] = ChatMessageHistory()
  return store[session_id]

with_message_history = RunnableWithMessageHistory(llm,get_message_history)

In [ ]:
config={"configurable":{"session_id":"chat1"}}

In [ ]:
!pip install langchain_classic

In [ ]:
from langchain_classic.chains.combine_documents.stuff import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

In [ ]:
document_chain = create_stuff_documents_chain(llm,prompt)
retriver = vector_store.as_retriever()
retriver_chain = create_retrieval_chain(retriver,document_chain)
# document_chain = llm|prompt
# retriver = vector_store.as_retriever()
# retriver_chain = retriver|document_chain

In [ ]:
response = with_message_history.invoke('what is llm',
                                       config=config)

In [ ]:
response.content

'LLM stands for Large Language Model. It refers to a type of artificial intelligence (AI) designed to process and understand human language at a large scale. LLMs are trained on vast amounts of text data, which enables them to learn patterns, relationships, and structures of language.\n\nThe primary goal of an LLM is to generate coherent and contextually relevant text based on a given input or prompt. These models can perform various tasks, such as:\n\n1. **Text generation**: Creating text based on a prompt, topic, or style.\n2. **Language translation**: Translating text from one language to another.\n3. **Text summarization**: Summarizing long pieces of text into shorter, more digestible versions.\n4. **Question answering**: Answering questions based on the context of a given text or conversation.\n5. **Conversational dialogue**: Engaging in natural-sounding conversations with humans.\n\nLLMs are typically trained using a range of techniques, including:\n\n1. **Masked language modelin

In [ ]:
response_2 = with_message_history.invoke('Data PreProcessing in llm',
                                         config=config)

In [ ]:
response_2.content

'Data Preprocessing in Large Language Models (LLMs) is a crucial step to prepare the text data for training and fine-tuning the model. Here are the common data preprocessing steps used in LLMs:\n\n### 1. **Text Cleaning**\n\n* **Removing special characters**: Removing special characters, such as punctuation, hashtags, and URLs, that are not essential for the model to learn.\n* **Removing stop words**: Removing common words like "the", "and", "a", etc. that do not add much value to the meaning of the text.\n* **Removing HTML tags**: Removing HTML tags and other markup language tags that are not relevant to the text.\n\n### 2. **Tokenization**\n\n* **Word-level tokenization**: Splitting text into individual words or tokens.\n* **Subword tokenization**: Splitting words into subwords or smaller units, such as wordpieces or BPE (Byte Pair Encoding) tokens.\n* **Character-level tokenization**: Splitting text into individual characters.\n\n### 3. **Normalization**\n\n* **Lowercasing**: Conver

In [ ]:
response_3 = with_message_history.invoke('what is my first question',
                                         config=config)

In [ ]:
response_3.content

'Your first question was: **"what is llm"**\n\nYou asked about LLM, which stands for Large Language Model, and I provided a detailed explanation of what LLMs are, how they work, and their applications.'

In [ ]:
response_4 = with_message_history.invoke('combine first and second answer',
                                         config=config)

In [ ]:
response_4.content

'Here\'s the combined answer:\n\n**What is LLM?**\n\nLLM stands for Large Language Model. It refers to a type of artificial intelligence (AI) designed to process and understand human language at a large scale. LLMs are trained on vast amounts of text data, which enables them to learn patterns, relationships, and structures of language.\n\nThe primary goal of an LLM is to generate coherent and contextually relevant text based on a given input or prompt. These models can perform various tasks, such as:\n\n1. **Text generation**: Creating text based on a prompt, topic, or style.\n2. **Language translation**: Translating text from one language to another.\n3. **Text summarization**: Summarizing long pieces of text into shorter, more digestible versions.\n4. **Question answering**: Answering questions based on the context of a given text or conversation.\n5. **Conversational dialogue**: Engaging in natural-sounding conversations with humans.\n\nLLMs are typically trained using a range of te

In [ ]:
config_2={"configurable":{"session_id":"chat2"}}

In [ ]:
def chatbot():
    while True:
        question = input("Ask your question: ")

        if question.lower() == "quit":
            print("Goodbye!")
            break

        response = with_message_history.invoke(
            {"input": question},
            config=config_2
        )

        print("AI:", response.content)

In [ ]:
chatbot()

Ask your question: what is llm
AI: LLM stands for Large Language Model. It refers to a type of artificial intelligence (AI) model that is designed to process and understand human language at a large scale. LLMs are trained on vast amounts of text data, which enables them to learn patterns, relationships, and structures of language.

Large Language Models are typically based on deep learning architectures, such as transformer models, and are trained using self-supervised learning techniques. This means that they learn to predict the next word in a sentence or to fill in missing words in a text, without being explicitly told what the correct answer is.

LLMs have many applications, including:

1. **Language translation**: LLMs can be used to translate text from one language to another.
2. **Text summarization**: LLMs can summarize long pieces of text into shorter, more digestible versions.
3. **Chatbots and conversational AI**: LLMs can be used to power chatbots and other conversational 